# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an end-to-end template for loading, exploring, and analyzing the [FAIR^2 mass spectrometry dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library.

### Dataset Source
The dataset is described via a [Croissant schema](https://mlcommons.org/croissant/) at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading

We will load the metadata and records from the dataset using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')  # Suppress typical pandas/np warnings for display

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)

# Print dataset overview (do not subscript metadata directly; use its attributes)
print(f"Name: {dataset.metadata.name}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Description: {dataset.metadata.description}")
print(f"Authors: {dataset.metadata.author}")
print(f"Published: {dataset.metadata.datePublished}")


## 2. Data Overview

Let's enumerate available record sets, their `@id`s, and associated fields.

> **Note:** To ensure unambiguous reference, all entities (record sets, fields, columns) are referenced by their Croissant `@id`.

In [ ]:
# List all record sets with their @ids
print("Available record sets (by @id):")
for recset in dataset.record_sets:
    print(f"  - @id: {recset.id}")
    print(f"    Name: {recset.name}")

    print("    Columns / Fields (@id):")
    for col in recset.columns:
        print(f"      - @id: {col.id}")
        print(f"        Name: {col.name}")
        print(f"        Data type: {col.data_type}")
    print("")

## 3. Data Extraction

We'll extract data from the main record set into a DataFrame for analysis. All field references use their `@id`.

- We'll identify typical numeric fields for analysis by examining the available fields above.

In [ ]:
# Collect all record set @ids
record_sets = [recset.id for recset in dataset.record_sets]
print("Extracting DataFrames for each record set...")
dataframes = {}

for recset in dataset.record_sets:
    print(f"- Loading: {recset.id} ({recset.name})")
    records = list(dataset.records(record_set=recset.id))
    dataframes[recset.id] = pd.DataFrame(records)

# Display all loaded record sets as DataFrame keys
print(f"Loaded DataFrames: {list(dataframes.keys())}")

# Select the main data table (usually has the most fields/rows)
main_record_set_id = record_sets[0]  # Adjust if needed after exploring step 2
print(f"Main record set @id: {main_record_set_id}")
print("Columns (Fields) in main table:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

We'll apply standard data processing steps, such as filtering on a specific numeric field, normalization, and grouping.

> All columns/fields are referenced by their Croissant `@id`. Adjust the variable assignments below as needed after viewing the actual overview in Step 2/3.

In [ ]:
# --- Step 1: Select a numeric field for analysis ---
# For this dataset, a typical field would be protein abundance or molecular weight. Adjust as needed after overview.
# Here we try to guess likely ones by matching their IDs/names.

# Example guesses based on common field names
main_df = dataframes[main_record_set_id]
numeric_candidates = [col for col in main_df.columns if any(word in col.lower() for word in ['abundance', 'mw', 'weight', 'count', 'number', 'quantity'])]
print("Likely numeric candidate columns:", numeric_candidates)

# Set the field to one of the candidates if available
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]  # Replace with actual field @id if needed
else:
    numeric_field_id = main_df.columns[0]  # Fallback

print(f"Using numeric field (by @id): {numeric_field_id}")

# --- Step 2: Filter records by threshold ---
threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 10
filtered_df = main_df[pd.to_numeric(main_df[numeric_field_id], errors='coerce') > threshold].copy()
print(f"Filtered: {len(filtered_df)} records with {numeric_field_id} > {threshold:.2f}")

# --- Step 3: Normalize the numeric field ---
filtered_df[f"{numeric_field_id}_normalized"] = (pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - \
                                                 pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()) / \
                                                pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()

print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# --- Step 4: Group by a categorical field ---
group_candidates = [col for col in main_df.columns if col != numeric_field_id and main_df[col].dtype == object]
if group_candidates:
    group_field_id = group_candidates[0]
    print(f"Grouping by field (by @id): {group_field_id}")
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(grouped_df.head())
else:
    group_field_id = None
    print("No suitable group field found.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and, when possible, its relationship to a group/categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric field
plt.figure(figsize=(8, 4))
sns.histplot(pd.to_numeric(main_df[numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If we have a group field, show mean by group
if group_field_id:
    plt.figure(figsize=(10, 5))
    sns.barplot(x=grouped_df[group_field_id], y=grouped_df[numeric_field_id])
    plt.title(f"Average {numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to use the `mlcroissant` library to:

- Load a [Croissant-structured FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) on mass spectrometry analysis.
- Explore the record set and field structure using Croissant `@id`s for traceability.
- Extract records, filter by a numeric field, normalize values, and group results.
- Visually explore data distributions and relationships.

This workflow can readily be extended using other record sets or fields as required by your research.